Microservices - REST (Synchron) - **Open Telemetry Variante**
---------------------------------------------------------

![](images/Microservices-REST.png)

Quelle: Buch Microservices Rezepte
- - -

Das Beispiel besteht aus drei Microservices: **Order**, **Customer** und **Catalog**. 

**Order** nutzt **Catalog** und **Customer** mit der REST-Schnittstelle. Ausserdem bietet jeder Microservice einige HTML-Seiten an.

Zusätzlich ist ein Service **Webshop** am laufen, der dem Benutzer mit einer Webseite einen einfachen Einstieg in das System ermöglicht.

- - -

Zuerst erstellen wir den Kubernetes Namespace ms-otel.

In [ ]:
! kubectl create namespace ms-otel

Jetzt ist ein guter Zeitpunkt um [HeadLamp](https://headlamp.dev/) (Nachfolger Kubernetes Dashboard) zu starten.

HeadLamp braucht einen Token welcher unter dem URL ausgegeben wird. Diesen markieren und mittels Ctrl+c kopieren und in der Headlamp Loginmaske wieder einfügen -> Ctrl+v.

In [ ]:
%%bash
echo "HeadLamp: http://"$(cat ~/data/server-ip)":30444"
kubectl create token headlamp-admin -n kube-system --duration=48h

Wir definieren eine OpenTelemetry-Instrumentation für den Namespace `ms-otel`, der die Python-Pods automatisch instrumentiert und ihre Traces per OTLP an den Collector sendet. 

Dabei werden Kontext-Propagation (W3C/B3) und Sampling konfiguriert, ohne dass Anpassungen am Anwendungscode nötig sind.D.h. die Services geben Trace-Informationen nach [W3C](https://www.w3.org/TR/trace-context/)/[B3](https://github.com/openzipkin/b3-propagation) weiter, damit Requests über mehrere Systeme verfolgbar sind.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: opentelemetry.io/v1alpha1
kind: Instrumentation
metadata:
  name: python-auto
  namespace: ms-otel
spec:
  exporter:
    endpoint: http://otel-collector-collector.opentelemetry.svc.cluster.local:4318

  propagators:
    - tracecontext
    - baggage

  sampler:
    type: parentbased_traceidratio
    argument: "0.5"

  python:
    image: ghcr.io/open-telemetry/opentelemetry-operator/autoinstrumentation-python:0.39b0
    env:
      - name: OTEL_EXPORTER_OTLP_PROTOCOL
        value: http/protobuf
      - name: OTEL_TRACES_EXPORTER
        value: otlp
      - name: OTEL_METRICS_EXPORTER
        value: none
      - name: OTEL_LOGS_EXPORTER
        value: none
      - name: OTEL_RESOURCE_ATTRIBUTES
        value: service.namespace=ms-otel
EOF


Wir starten unsere Microservices.

Damit OpenTelemetry diese erkennt und automatisch instrumentiert wurde die Annotation `instrumentation.opentelemetry.io/inject-python` und `prometheus.io` hinzugefügt:

      template:
        metadata:
          labels:
            app: catalog
          annotations:
            instrumentation.opentelemetry.io/inject-python: "python-auto"
            prometheus.io/scrape: "true"
            prometheus.io/path: "/metrics"
            prometheus.io/port: "8080"

Für eine volle Unterstützung von OpenTelemetry sind programmtechnische Erweiterungen nötig:
* [Language APIs & SDKs](https://opentelemetry.io/docs/languages/)

Und für Prometheus Metrics Daten:
* [Client libraries](https://prometheus.io/docs/instrumenting/clientlibs/)

**Hinweis**: nach diesen Erweiterungen kann die Annotation `instrumentation.opentelemetry.io/inject-python` entfernt werden.

In [ ]:
%%bash
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-0-0-deployment/catalog-deployment.yaml
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-0-0-deployment/customer-deployment.yaml
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-0-0-deployment/order-deployment.yaml
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-0-0-deployment/webshop-deployment.yaml 
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/catalog-service.yaml
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/customer-service.yaml
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/order-service.yaml
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/webshop-service.yaml
kubectl get pod,services --namespace ms-otel

Da wir keinen LoadBalancer haben müssen wir mit einem kleinen Shellscript selber die IP des Clusters und der gemappte Port (port-based-routing) als URL aufbereiten.

In [ ]:
! echo "http://"$(cat ~/data/server-ip)":"$(kubectl get service --namespace ms-otel webshop -o=jsonpath='{ .spec.ports[0].nodePort }')/webshop

---
## OpenTelemetry Observability Stack (Jaeger mit Monitoring-Stack: Prometheus, Grafana, AlertManager)

![](https://opentelemetry.io/img/otel-diagram.svg)

Quelle: [Opentelemetry](https://opentelemetry.io/docs/)

---

Traces laufen über den OpenTelemetry Collector nach Jaeger.

Opentelemetry scrappt die Metriken von den Microservices und leitet sie an Prometheus weiter.

Grafana mit seinen vorkonfigurierten Dashboard liest die Daten aus Prometheus und Jaeger und zeigt diese an.


In [ ]:
%%bash
NAMESPACE="opentelemetry"
SERVER_IP="$(cat ~/data/server-ip 2>/dev/null || true)"
JAEGER_PORT="$(kubectl -n "${NAMESPACE}" get svc jaeger -o=jsonpath='{.spec.ports[?(@.port==16686)].nodePort}')"
PROMETHEUS_PORT="$(kubectl -n "${NAMESPACE}" get svc prometheus-prometheus -o=jsonpath='{.spec.ports[?(@.port==9090)].nodePort}')"
ALERTMANAGER_PORT="$(kubectl -n "${NAMESPACE}" get svc prometheus-alertmanager -o=jsonpath='{.spec.ports[?(@.port==9093)].nodePort}')"
GRAFANA_PORT="$(kubectl -n "${NAMESPACE}" get svc prometheus-grafana -o=jsonpath='{.spec.ports[?(@.port==80)].nodePort}')"

echo "Jaeger UI       : http://${SERVER_IP}:${JAEGER_PORT}"
echo "Prometheus UI   : http://${SERVER_IP}:${PROMETHEUS_PORT}"
echo "Alertmanager UI : http://${SERVER_IP}:${ALERTMANAGER_PORT}"
echo "Grafana UI      : http://${SERVER_IP}:${GRAFANA_PORT}"

---

### Jaeger

Jaeger visualisiert die (Netzwerk-)Kommunikation zwischen Microservices und stellt deren Latenzen dar.

Wähle **Search → Microservices** und gib `order` ein, anschliessend **Find Traces** ausführen.
Alternativ kannst du zur **System Architecture** wechseln, um eine Gesamtübersicht der Service-Struktur zu erhalten.

**Um die Verbindungen sichtbar zu machen, erzeugen wir ein wenig Traffic**

Argumente
* -t 10s = Dauer in Sekunden
* -c 60  = Anzahl gleichzeitige Requests
    

In [ ]:
%%bash
sudo apt-get install -y apache2-utils -y
URL="http://"$(cat ~/data/server-ip)":"$(kubectl get service --namespace ms-otel webshop -o=jsonpath='{ .spec.ports[0].nodePort }')/webshop
ab -t 60s -c 10  ${URL}/order/order

### Sequenzdiagramm

Dieses Script lädt einen Trace direkt aus Jaeger und erzeugt daraus automatisch ein [Mermaid](https://mermaid.ai/web/) Sequenzdiagramm.

Das Ergebnis steht in der Datei [sequence.mmd](sequence.mmd)

**Wichtig**: Die Variable TRACE_ID muss mit einer gültigen Jaeger Trace-ID gesetzt werden.

In [ ]:
import subprocess
import requests

# Namespace
NAMESPACE = "opentelemetry"

# SERVER_IP lesen
server_ip = subprocess.check_output(
    "cat ~/data/server-ip",
    shell=True,
    text=True
).strip()

# NodePort von Jaeger lesen
jaeger_port = subprocess.check_output(
    f"kubectl -n {NAMESPACE} get svc jaeger "
    "-o=jsonpath='{.spec.ports[?(@.port==16686)].nodePort}'",
    shell=True,
    text=True
).strip()

# URL bauen
JAEGER_URL = f"http://{server_ip}:{jaeger_port}"

print("Jaeger URL:", JAEGER_URL)

# Trace ID einsetzen
TRACE_ID = "0630d13d3c23e60171ed974acc0f1f98"

# Trace laden
response = requests.get(f"{JAEGER_URL}/api/traces/{TRACE_ID}")
response.raise_for_status()

data = response.json()
trace = data["data"][0]

processes = trace["processes"]
spans = trace["spans"]

span_by_id = {span["spanID"]: span for span in spans}


def service_name(span):
    return processes[span["processID"]]["serviceName"]


def operation_name(span):
    return span.get("operationName", "unknown")


def parent_span(span):
    for ref in span.get("references", []):
        if ref.get("refType") == "CHILD_OF":
            return span_by_id.get(ref.get("spanID"))
    return None


def is_relevant(span):
    op = operation_name(span)

    if op in {"GET", "POST", "PUT", "DELETE", "PATCH"}:
        return False

    if op.startswith("jinja2."):
        return False

    return True


edges = []

for span in spans:
    if not is_relevant(span):
        continue

    parent = parent_span(span)

    if not parent:
        continue

    parent_service = service_name(parent)
    child_service = service_name(span)

    if parent_service == child_service:
        continue

    edges.append({
        "startTime": span.get("startTime", 0),
        "parent": parent_service,
        "child": child_service,
        "operation": operation_name(span),
    })

# Nach Zeit sortieren
edges = sorted(edges, key=lambda e: e["startTime"])

# Mermaid erzeugen
lines = ["sequenceDiagram"]

seen = set()

for edge in edges:
    key = (edge["parent"], edge["child"], edge["operation"])

    if key in seen:
        continue

    seen.add(key)

    lines.append(
        f'    {edge["parent"]}->>{edge["child"]}: {edge["operation"]}'
    )

mermaid = "\n".join(lines)

print("\n--- Mermaid ---\n")
print(mermaid)

# Datei schreiben
with open("sequence.mmd", "w", encoding="utf-8") as f:
    f.write(mermaid)

print("\nsequence.mmd geschrieben")

---

### Prometheus

Im User Interface lassen sich die überwachten Targets sowie deren Status einsehen.

Gib dazu die Query `up` ein oder navigiere über **Status → Targets**, um die überwachten Microservices anzuzeigen.

Zusätzlich können unter anderem folgende Metriken ausgewertet werden:

* `http_requests_total{method="GET"}` – Anzahl HTTP-GET-Aufrufe
* `process_memory_usage_bytes` – Speicherverbrauch pro Pod
* `up{namespace="ms-otel"}` – alle aktiven Pods im Namespace `ms-otel`


### Grafana Dashboard

Das Grafana Dashboard enthält mehrere vordefinierte Dashboards, darunter:

* **Kubernetes / Compute Resources / Namespace (Pods)** – zeigt den CPU-Verbrauch der Pods innerhalb eines Namespace
* **Kubernetes / Networking / Namespace (Pods)** – zeigt den Netzwerkverkehr der Pods innerhalb eines Namespace


---
### Alerting mit Prometheus und Alertmanager

Alerts werden in Prometheus definiert und automatisch ausgelöst, sobald eine festgelegte Bedingung erfüllt ist. Diese Alerts werden anschliessend an den Prometheus Alertmanager weitergeleitet, der für die Darstellung, Gruppierung und Weiterleitung (z. B. E-Mail oder Slack) zuständig ist.

Für das Verständnis ist wichtig: Ein Alert erscheint im Alertmanager nur dann, wenn er in Prometheus tatsächlich den Status **firing** erreicht hat.

**1. Prometheus – Regeln und Status**
* **Status → Rules** → zeigt alle geladenen Alert-Regeln und deren Zustand (*inactive*, *pending*, *firing*)
* **Alerts** → zeigt aktuell aktive Alerts (nur *firing*)

**2. Alertmanager – Darstellung der Alerts**
* **Alerts** → alle aktuell empfangenen Alerts vom Prometheus
* **Silences** → unterdrückte Alerts (z. B. während Wartung)
* **Status** → Routing- und Konfigurationsübersicht


In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: monitoring.coreos.com/v1
kind: PrometheusRule
metadata:
  name: ms-otel-alerts
  namespace: ms-otel
  labels:
    release: prometheus
spec:
  groups:
    - name: ms-otel.rules
      rules:
        - alert: MSOTelTargetDown
          expr: up{namespace="ms-otel"} == 0
          for: 10s
          labels:
            severity: critical
          annotations:
            summary: "Target nicht erreichbar"
            description: "{{ \$labels.job }} / {{ \$labels.instance }} ist nicht erreichbar."

        - alert: MSOTelAnyHttpTraffic
          expr: rate(http_requests_total{namespace="ms-otel"}[1m]) > 0
          for: 10s
          labels:
            severity: info
          annotations:
            summary: "HTTP-Traffic erkannt"
            description: "Service {{ \$labels.service }} verarbeitet HTTP-Requests."

        - alert: MSOTelAnyMemoryUsage
          expr: process_memory_usage_bytes{namespace="ms-otel"} > 1
          for: 10s
          labels:
            severity: warning
          annotations:
            summary: "Speicherverbrauch erkannt"
            description: "{{ \$labels.pod }} verwendet Speicher."

        - alert: MSOTelPodRestarted
          expr: increase(kube_pod_container_status_restarts_total{namespace="ms-otel"}[5m]) > 0
          for: 10s
          labels:
            severity: warning
          annotations:
            summary: "Pod wurde neu gestartet"
            description: "{{ \$labels.pod }} wurde innerhalb der letzten 5 Minuten neu gestartet."
EOF

- - -

Aufräumen

In [ ]:
! kubectl delete namespace ms-otel

- - -
### Quellen

* Sourcecode: https://gitlab.com/ch-mc-b/autoshop-ms/app/shop/-/tree/v3.0.0?ref_type=heads
* Kubernetes Deklarationen: https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates
* Container Registry: https://gitlab.com/ch-mc-b/autoshop-ms/app/shop/container_registry